# Paper Replication: Naive Data Leakage & Transfer Learning

## Overview
This notebook replicates the paper's methodology to demonstrate:
- **Phase 1**: The "Naive" 80/20 Random Split (where leakage happens)
- **Phase 2**: Standard preprocessing (one-hot encoding + Z-standardization)
- **Phase 3**: Cascading prediction strategy (10 1/s → 1 & 100 1/s)
- **Phase 4**: Baseline models on combined dataset (showing high accuracy from replicate leakage)
- **Phase 5**: Transfer Learning experiment (Batch 1 → Batch 2 with covariate shift + LightGBM recovery)

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

RANDOM_STATE = 42

# Column definitions
TARGET_1 = "Viscosity_at_shear_rate_1_1/s"
TARGET_10 = "Viscosity_at_shear_rate_10_1/s"
TARGET_100 = "Viscosity_at_shear_rate_100_1/s"
TARGET_COLS = [TARGET_1, TARGET_10, TARGET_100]

CAT_COLS = ["Formulation", "Dispersent_Type"]
CONT_COLS = ["Solid_Content_pct", "Solid_Additive_pct"]
BATCH_COL = "Source_Batch"

DROP_COLS = ["Composite_Mix_ID"]

# Load combined raw data
DATA_PATH = Path("../data/processed/combined_slurry_data_cleaned.csv")
df = pd.read_csv(DATA_PATH).drop(columns=DROP_COLS)
print(f"Total rows in combined dataset: {len(df)}")
print(f"Batch composition: {df[BATCH_COL].value_counts().to_dict()}")

#print(df.head())


Total rows in combined dataset: 178
Batch composition: {'Batch_1': 91, 'Batch_2': 87}


In [9]:
from sklearn.model_selection import KFold

# =============================================================================
# PHASE 1: Naive 80/20 Split (Paper Replication)
# =============================================================================
# Random split on COMBINED data (Batch 1 + Batch 2), treating each row as an
# independent sample. This reproduces the paper's high baseline setting.

print("\n" + "="*80)
print("PHASE 1: Naive 80/20 Random Split on Combined Dataset")
print("="*80)

idx_all = np.arange(len(df))
N_SPLITS = 5

# Naive random 80/20 split on all rows
idx_train_naive, idx_test_naive = train_test_split(
    idx_all,
    test_size=0.2,
    random_state=RANDOM_STATE,
    shuffle=True,
)

# Fixed-seed K-fold CV on the TRAIN partition only
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
cv_splits_train = []

for fold_id, (train_rel, valid_rel) in enumerate(kf.split(idx_train_naive), start=1):
    train_idx = idx_train_naive[train_rel]
    valid_idx = idx_train_naive[valid_rel]
    cv_splits_train.append((train_idx, valid_idx))

print(f"Using naive 80/20 random split with seed={RANDOM_STATE}")
print(f"Using fixed {N_SPLITS}-fold CV on training partition")
print("Active split in this cell: global train vs global test")

df_train_naive = df.iloc[idx_train_naive].copy()
df_test_naive = df.iloc[idx_test_naive].copy()

print(f"Train set size: {len(df_train_naive)} rows")
print(f"Test set size: {len(df_test_naive)} rows")
print(f"Batch composition in train: {df_train_naive[BATCH_COL].value_counts().to_dict()}")
print(f"Batch composition in test: {df_test_naive[BATCH_COL].value_counts().to_dict()}")


PHASE 1: Naive 80/20 Random Split on Combined Dataset
Using naive 80/20 random split with seed=42
Using fixed 5-fold CV on training partition
Active split in this cell: global train vs global test
Train set size: 142 rows
Test set size: 36 rows
Batch composition in train: {'Batch_2': 71, 'Batch_1': 71}
Batch composition in test: {'Batch_1': 20, 'Batch_2': 16}


In [10]:
# =============================================================================
# PHASE 2: Data Preprocessing (Standard Approach)
# =============================================================================
# Categorical: One-hot encoding
# Continuous: Z-standardization (fit on train, apply to test)

print("\n" + "="*80)
print("PHASE 2: Data Preprocessing")
print("="*80)

# One-hot encode categorical features (fit on train, apply to test)
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
train_cat = ohe.fit_transform(df_train_naive[CAT_COLS])
test_cat = ohe.transform(df_test_naive[CAT_COLS])

cat_feature_names = ohe.get_feature_names_out(CAT_COLS)
train_cat_df = pd.DataFrame(train_cat, columns=cat_feature_names, index=df_train_naive.index)
test_cat_df = pd.DataFrame(test_cat, columns=cat_feature_names, index=df_test_naive.index)

# Z-standardization (fit on train, apply to test)
scaler = StandardScaler()
train_cont = scaler.fit_transform(df_train_naive[CONT_COLS])
test_cont = scaler.transform(df_test_naive[CONT_COLS])

train_cont_df = pd.DataFrame(train_cont, columns=CONT_COLS, index=df_train_naive.index)
test_cont_df = pd.DataFrame(test_cont, columns=CONT_COLS, index=df_test_naive.index)

# Combine features
X_train_naive = pd.concat([train_cat_df, train_cont_df], axis=1)
X_test_naive = pd.concat([test_cat_df, test_cont_df], axis=1)

y_train_naive = df_train_naive[TARGET_COLS].copy()
y_test_naive = df_test_naive[TARGET_COLS].copy()

print(f"Feature count: {X_train_naive.shape[1]}")
print(f"Features: {list(X_train_naive.columns)}")
print(f"✅ Preprocessing complete")


PHASE 2: Data Preprocessing
Feature count: 10
Features: ['Formulation_B', 'Formulation_F1', 'Formulation_F2', 'Formulation_F3', 'Dispersent_Type_Hypermer', 'Dispersent_Type_Tego', 'Dispersent_Type_Triton', 'Dispersent_Type_none', 'Solid_Content_pct', 'Solid_Additive_pct']
✅ Preprocessing complete


In [11]:
# =============================================================================
# PHASE 3: Cascading Prediction Strategy
# =============================================================================
# Step 1: Train a model to predict Viscosity at 10 1/s
# Step 2: Use the predicted 10 1/s as a 5th feature
# Step 3: Train separate models for 1 1/s and 100 1/s

print("\n" + "="*80)
print("PHASE 3: Cascading Strategy Definition")
print("="*80)
print("""
Cascading Approach:
  1. Train model to predict Viscosity @ 10 1/s using 4 input features
  2. Append predicted 10 1/s as 5th feature to dataset
  3. Train separate models for 1 1/s and 100 1/s using all 5 features
  
This leverages the correlation between shear rates to improve predictions.
""")

def regression_metrics(y_true, y_pred):
    """Compute R², RMSE, MAE."""
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
    }


def print_metrics_table(title, metrics_dict, split_info=""):
    """Print formatted metrics table."""
    rows = []
    for target_name, metrics in metrics_dict.items():
        rows.append({
            "Target": target_name,
            "R²": metrics["R2"],
            "RMSE": metrics["RMSE"],
            "MAE": metrics["MAE"],
        })
    out_df = pd.DataFrame(rows)
    print(f"\n{title}")
    if split_info:
        print(f"  {split_info}")
    print(out_df.to_string(index=False))
    return out_df


PHASE 3: Cascading Strategy Definition

Cascading Approach:
  1. Train model to predict Viscosity @ 10 1/s using 4 input features
  2. Append predicted 10 1/s as 5th feature to dataset
  3. Train separate models for 1 1/s and 100 1/s using all 5 features

This leverages the correlation between shear rates to improve predictions.



In [12]:
# =============================================================================
# PHASE 4: Baseline Models on Combined Dataset (Naive 80/20 Split)
# =============================================================================
# This reproduces the paper's high-R2 baseline setting using row-wise random
# splitting and cascading predictions.

print("\n" + "="*80)
print("PHASE 4: Baseline Models (Combined Dataset, Naive 80/20 Split)")
print("="*80)

# XGBoost hyperparameters (paper-aligned)
xgb_params = {
    "learning_rate": 0.1,
    "max_depth": 3,
    "min_child_weight": 1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "gamma": 0.1,
    "n_estimators": 100,
    "objective": "reg:squarederror",
    "random_state": RANDOM_STATE,
}

# Random Forest hyperparameters (paper-aligned)
rf_params = {
    "n_estimators": 100,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}


def preprocess_split(df_train_split, df_valid_split):
    """Fit preprocessing on train split only, transform train/valid."""
    ohe_split = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    scaler_split = StandardScaler()

    Xtr_cat = ohe_split.fit_transform(df_train_split[CAT_COLS])
    Xva_cat = ohe_split.transform(df_valid_split[CAT_COLS])

    cat_names = ohe_split.get_feature_names_out(CAT_COLS)
    Xtr_cat_df = pd.DataFrame(Xtr_cat, columns=cat_names, index=df_train_split.index)
    Xva_cat_df = pd.DataFrame(Xva_cat, columns=cat_names, index=df_valid_split.index)

    Xtr_cont = scaler_split.fit_transform(df_train_split[CONT_COLS])
    Xva_cont = scaler_split.transform(df_valid_split[CONT_COLS])

    Xtr_cont_df = pd.DataFrame(Xtr_cont, columns=CONT_COLS, index=df_train_split.index)
    Xva_cont_df = pd.DataFrame(Xva_cont, columns=CONT_COLS, index=df_valid_split.index)

    X_train_split = pd.concat([Xtr_cat_df, Xtr_cont_df], axis=1)
    X_valid_split = pd.concat([Xva_cat_df, Xva_cont_df], axis=1)

    y_train_split = df_train_split[TARGET_COLS].copy()
    y_valid_split = df_valid_split[TARGET_COLS].copy()
    return X_train_split, X_valid_split, y_train_split, y_valid_split


def run_cascade_eval(model_ctor, model_params, X_train, X_eval, y_train):
    """Train cascade and return predictions for eval set."""
    # Step 1: predict 10 1/s
    m10 = model_ctor(**model_params)
    m10.fit(X_train, y_train[TARGET_10])

    pred10_train = m10.predict(X_train)
    pred10_eval = m10.predict(X_eval)

    # Step 2: add predicted 10 1/s as feature
    X_train_c = X_train.copy()
    X_eval_c = X_eval.copy()
    X_train_c["Predicted_Viscosity_10_1/s"] = pred10_train
    X_eval_c["Predicted_Viscosity_10_1/s"] = pred10_eval

    # Step 3: predict 1 and 100 1/s
    m1 = model_ctor(**model_params)
    m100 = model_ctor(**model_params)
    m1.fit(X_train_c, y_train[TARGET_1])
    m100.fit(X_train_c, y_train[TARGET_100])

    pred1_eval = m1.predict(X_eval_c)
    pred100_eval = m100.predict(X_eval_c)

    return pred10_eval, pred1_eval, pred100_eval


# ---------------- CV on training partition ----------------
cv_rows = []
for fold_no, (idx_tr, idx_va) in enumerate(cv_splits_train, start=1):
    df_tr = df.iloc[idx_tr].copy()
    df_va = df.iloc[idx_va].copy()

    Xtr, Xva, ytr, yva = preprocess_split(df_tr, df_va)

    # XGBoost CV metrics
    p10, p1, p100 = run_cascade_eval(XGBRegressor, xgb_params, Xtr, Xva, ytr)
    cv_rows.extend([
        {"model": "XGBoost", "fold": fold_no, "target": "10 1/s", **regression_metrics(yva[TARGET_10], p10)},
        {"model": "XGBoost", "fold": fold_no, "target": "1 1/s", **regression_metrics(yva[TARGET_1], p1)},
        {"model": "XGBoost", "fold": fold_no, "target": "100 1/s", **regression_metrics(yva[TARGET_100], p100)},
    ])

    # Random Forest CV metrics
    p10, p1, p100 = run_cascade_eval(RandomForestRegressor, rf_params, Xtr, Xva, ytr)
    cv_rows.extend([
        {"model": "Random Forest", "fold": fold_no, "target": "10 1/s", **regression_metrics(yva[TARGET_10], p10)},
        {"model": "Random Forest", "fold": fold_no, "target": "1 1/s", **regression_metrics(yva[TARGET_1], p1)},
        {"model": "Random Forest", "fold": fold_no, "target": "100 1/s", **regression_metrics(yva[TARGET_100], p100)},
    ])

cv_df = pd.DataFrame(cv_rows)
cv_summary = (
    cv_df.groupby(["model", "target"])[["R2", "RMSE", "MAE"]]
    .mean()
    .reset_index()
)

print("\nCross-validation mean metrics (on training partition):")
print(cv_summary.to_string(index=False))

# ---------------- Final fit on naive train, evaluate on naive test ----------------

# XGBoost final
t10_xgb, t1_xgb, t100_xgb = run_cascade_eval(
    XGBRegressor,
    xgb_params,
    X_train_naive,
    X_test_naive,
    y_train_naive,
)

xgb_metrics = {
    "10 1/s": regression_metrics(y_test_naive[TARGET_10], t10_xgb),
    "1 1/s": regression_metrics(y_test_naive[TARGET_1], t1_xgb),
    "100 1/s": regression_metrics(y_test_naive[TARGET_100], t100_xgb),
}

xgb_df = print_metrics_table(
    "XGBoost with Cascading (Naive Hold-out Test Set)",
    xgb_metrics,
    split_info=f"Train: {len(X_train_naive)} | Test: {len(X_test_naive)}"
)
xgb_df["model"] = "XGBoost"

# Random Forest final
t10_rf, t1_rf, t100_rf = run_cascade_eval(
    RandomForestRegressor,
    rf_params,
    X_train_naive,
    X_test_naive,
    y_train_naive,
)

rf_metrics = {
    "10 1/s": regression_metrics(y_test_naive[TARGET_10], t10_rf),
    "1 1/s": regression_metrics(y_test_naive[TARGET_1], t1_rf),
    "100 1/s": regression_metrics(y_test_naive[TARGET_100], t100_rf),
}

rf_df = print_metrics_table(
    "Random Forest with Cascading (Naive Hold-out Test Set)",
    rf_metrics,
    split_info=f"Train: {len(X_train_naive)} | Test: {len(X_test_naive)}"
)
rf_df["model"] = "Random Forest"

print("\nObservation: Naive combined split can inflate apparent generalization due to replicate leakage.")


PHASE 4: Baseline Models (Combined Dataset, Naive 80/20 Split)

Cross-validation mean metrics (on training partition):
        model  target       R2      RMSE       MAE
Random Forest   1 1/s 0.570391 41.887981 27.185672
Random Forest  10 1/s 0.814103  4.697856  3.117331
Random Forest 100 1/s 0.858634  1.139160  0.794985
      XGBoost   1 1/s 0.585638 41.453855 26.720785
      XGBoost  10 1/s 0.831403  4.483226  3.055519
      XGBoost 100 1/s 0.864234  1.120526  0.786957

XGBoost with Cascading (Naive Hold-out Test Set)
  Train: 142 | Test: 36
 Target       R²      RMSE       MAE
 10 1/s 0.873784  3.391959  2.250005
  1 1/s 0.590625 29.180476 16.430838
100 1/s 0.875944  1.045171  0.639761

Random Forest with Cascading (Naive Hold-out Test Set)
  Train: 142 | Test: 36
 Target       R²      RMSE       MAE
 10 1/s 0.877687  3.339104  2.174583
  1 1/s 0.614650 28.311260 16.143720
100 1/s 0.876827  1.041442  0.611319

Observation: Naive combined split can inflate apparent generalization du

In [13]:
# =============================================================================
# PHASE 5: Transfer Learning Experiment (Batch 1 -> Batch 2)
# =============================================================================
# Step A: Train base model on Batch 1 (source domain)
# Step B: Test directly on Batch 2 (target domain) -> shows covariate shift
# Step C: Apply LightGBM update with 20% of Batch 2 -> shows recovery

print("\n" + "="*80)
print("PHASE 5: Transfer Learning (Batch 1 as Source, Batch 2 as Target)")
print("="*80)

# Separate by batch
batch1_df = df[df[BATCH_COL] == "Batch_1"].copy()
batch2_df = df[df[BATCH_COL] == "Batch_2"].copy()

print(f"\nBatch 1 (source domain): {len(batch1_df)} samples")
print(f"Batch 2 (target domain): {len(batch2_df)} samples")

# Preprocessing for Batch 1 (as source domain)
ohe_batch = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
batch1_cat = ohe_batch.fit_transform(batch1_df[CAT_COLS])
batch2_cat = ohe_batch.transform(batch2_df[CAT_COLS])

batch1_cat_df = pd.DataFrame(batch1_cat, columns=ohe_batch.get_feature_names_out(CAT_COLS), index=batch1_df.index)
batch2_cat_df = pd.DataFrame(batch2_cat, columns=ohe_batch.get_feature_names_out(CAT_COLS), index=batch2_df.index)

# Scaler fit only on Batch 1
scaler_batch = StandardScaler()
batch1_cont = scaler_batch.fit_transform(batch1_df[CONT_COLS])
batch2_cont = scaler_batch.transform(batch2_df[CONT_COLS])

batch1_cont_df = pd.DataFrame(batch1_cont, columns=CONT_COLS, index=batch1_df.index)
batch2_cont_df = pd.DataFrame(batch2_cont, columns=CONT_COLS, index=batch2_df.index)

# Combine features
X_batch1 = pd.concat([batch1_cat_df, batch1_cont_df], axis=1)
X_batch2 = pd.concat([batch2_cat_df, batch2_cont_df], axis=1)

y_batch1 = batch1_df[TARGET_COLS].copy()
y_batch2 = batch2_df[TARGET_COLS].copy()

print(f"\nBatch 1 features: {X_batch1.shape}")
print(f"Batch 2 features: {X_batch2.shape}")

# ---- Step A: Train Base Models on Batch 1 ----
print("\n--- Step A: Base Training on Batch 1 (Source) ---")

lgb_base_params = {
    "learning_rate": 0.10,
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "verbosity": -1,
}

# Base model for 10 1/s
lgb_10_base = LGBMRegressor(**lgb_base_params)
lgb_10_base.fit(X_batch1, y_batch1[TARGET_10])

pred10_batch2_base = lgb_10_base.predict(X_batch2)

covariate_shift_metrics_10 = regression_metrics(y_batch2[TARGET_10], pred10_batch2_base)
print(f"Base LightGBM (10 1/s) tested directly on Batch 2: R2 = {covariate_shift_metrics_10['R2']:.3f}")

# ---- Step B: Show Covariate Shift (No TL) ----
print("\n--- Step B: Direct Test on Batch 2 (No Transfer Learning) ---")

X_batch2_cascade_base = X_batch2.copy()
X_batch2_cascade_base["Predicted_Viscosity_10_1/s"] = pred10_batch2_base

# Train models for 1 and 100 on Batch 1 with cascading
X_batch1_cascade = X_batch1.copy()
pred10_batch1_base = lgb_10_base.predict(X_batch1)
X_batch1_cascade["Predicted_Viscosity_10_1/s"] = pred10_batch1_base

lgb_1_base = LGBMRegressor(**lgb_base_params)
lgb_100_base = LGBMRegressor(**lgb_base_params)

lgb_1_base.fit(X_batch1_cascade, y_batch1[TARGET_1])
lgb_100_base.fit(X_batch1_cascade, y_batch1[TARGET_100])

pred1_batch2_base = lgb_1_base.predict(X_batch2_cascade_base)
pred100_batch2_base = lgb_100_base.predict(X_batch2_cascade_base)

no_tl_metrics = {
    "10 1/s": regression_metrics(y_batch2[TARGET_10], pred10_batch2_base),
    "1 1/s": regression_metrics(y_batch2[TARGET_1], pred1_batch2_base),
    "100 1/s": regression_metrics(y_batch2[TARGET_100], pred100_batch2_base),
}

no_tl_df = print_metrics_table(
    "No Transfer Learning (Base Batch 1 models tested on Batch 2)",
    no_tl_metrics,
    split_info="Shows covariate shift drop from source to target domain"
)
no_tl_df["approach"] = "No_TL_Baseline"

# ---- Step C: Transfer Learning with LightGBM Update ----
print("\n--- Step C: Transfer Learning with LightGBM Update ---")
print("""
Fine-tuning Strategy:
  - Base models from Batch 1 are kept fixed via init_model
  - New trees are appended and adapted using Batch 2 data
  - Original 500-tree ensemble is preserved
  - Added trees (100) use a lower learning rate (0.05)
""")

# Split Batch 2: 20% for fine-tuning, 80% for final test
idx_batch2 = np.arange(len(X_batch2))
idx_finetune, idx_finaltest = train_test_split(
    idx_batch2,
    test_size=0.8,
    random_state=RANDOM_STATE,
)

X_batch2_tune = X_batch2.iloc[idx_finetune].copy()
X_batch2_test = X_batch2.iloc[idx_finaltest].copy()
y_batch2_tune = y_batch2.iloc[idx_finetune].copy()
y_batch2_test = y_batch2.iloc[idx_finaltest].copy()

print(f"Batch 2 split: fine-tune {len(X_batch2_tune)} (20%) | test {len(X_batch2_test)} (80%)")

# LightGBM update params
lgb_update_params = {
    "learning_rate": 0.05,
    "n_estimators": 100,
    "random_state": RANDOM_STATE,
    "verbosity": -1,
}

# Update 10 1/s model
lgb_10_updated = LGBMRegressor(**lgb_update_params)
lgb_10_updated.fit(
    X_batch2_tune,
    y_batch2_tune[TARGET_10],
    init_model=lgb_10_base.booster_,
)

pred10_batch2_tune = lgb_10_updated.predict(X_batch2_tune)
pred10_batch2_test = lgb_10_updated.predict(X_batch2_test)

# Generate cascading feature using updated model
X_batch2_tune_cascade = X_batch2_tune.copy()
X_batch2_test_cascade = X_batch2_test.copy()
X_batch2_tune_cascade["Predicted_Viscosity_10_1/s"] = pred10_batch2_tune
X_batch2_test_cascade["Predicted_Viscosity_10_1/s"] = pred10_batch2_test

# Update 1 and 100 models with fine-tuning data
lgb_1_updated = LGBMRegressor(**lgb_update_params)
lgb_100_updated = LGBMRegressor(**lgb_update_params)

lgb_1_updated.fit(
    X_batch2_tune_cascade,
    y_batch2_tune[TARGET_1],
    init_model=lgb_1_base.booster_,
)
lgb_100_updated.fit(
    X_batch2_tune_cascade,
    y_batch2_tune[TARGET_100],
    init_model=lgb_100_base.booster_,
)

# Evaluate on test set
pred1_batch2_test = lgb_1_updated.predict(X_batch2_test_cascade)
pred100_batch2_test = lgb_100_updated.predict(X_batch2_test_cascade)

tl_metrics = {
    "10 1/s": regression_metrics(y_batch2_test[TARGET_10], pred10_batch2_test),
    "1 1/s": regression_metrics(y_batch2_test[TARGET_1], pred1_batch2_test),
    "100 1/s": regression_metrics(y_batch2_test[TARGET_100], pred100_batch2_test),
}

tl_df = print_metrics_table(
    "Transfer Learning (LightGBM with Update, tested on Batch 2 hold-out)",
    tl_metrics,
    split_info=f"Fine-tune: {len(X_batch2_tune)} (20%) | Test: {len(X_batch2_test)} (80%)"
)
tl_df["approach"] = "TL_LightGBM_Update"

print("\nTransfer Learning complete")
print("\nComparison (No-TL vs TL on held-out Batch 2 test set):")
for target in ["10 1/s", "1 1/s", "100 1/s"]:
    no_tl_r2 = no_tl_metrics[target]["R2"]
    tl_r2 = tl_metrics[target]["R2"]
    delta = tl_r2 - no_tl_r2
    print(f"  {target}: No-TL R2 = {no_tl_r2:.3f}, TL R2 = {tl_r2:.3f}, delta = {delta:+.3f}")


PHASE 5: Transfer Learning (Batch 1 as Source, Batch 2 as Target)

Batch 1 (source domain): 91 samples
Batch 2 (target domain): 87 samples

Batch 1 features: (91, 9)
Batch 2 features: (87, 9)

--- Step A: Base Training on Batch 1 (Source) ---
Base LightGBM (10 1/s) tested directly on Batch 2: R2 = 0.155

--- Step B: Direct Test on Batch 2 (No Transfer Learning) ---

No Transfer Learning (Base Batch 1 models tested on Batch 2)
  Shows covariate shift drop from source to target domain
 Target        R²      RMSE       MAE
 10 1/s  0.154989 10.553342  8.096571
  1 1/s -1.197551 75.426931 56.672085
100 1/s  0.540371  2.298842  1.754441

--- Step C: Transfer Learning with LightGBM Update ---

Fine-tuning Strategy:
  - Base models from Batch 1 are kept fixed via init_model
  - New trees are appended and adapted using Batch 2 data
  - Original 500-tree ensemble is preserved
  - Added trees (100) use a lower learning rate (0.05)

Batch 2 split: fine-tune 17 (20%) | test 70 (80%)

Transfer Lea

In [14]:
# =============================================================================
# SUMMARY: Combined Results
# =============================================================================

print("\n" + "="*80)
print("SUMMARY: Paper Replication Results")
print("="*80)

# Combine all results for comparison
all_results = pd.concat([
    xgb_df.assign(approach="Baseline_XGBoost_Combined_Naive80_20"),
    rf_df.assign(approach="Baseline_RF_Combined_Naive80_20"),
    no_tl_df,
    tl_df,
], ignore_index=True)

print("\nComplete Results Table:")
print(all_results[["Target", "approach", "R²", "RMSE", "MAE"]].to_string(index=False))

print("\n" + "="*80)
print("KEY FINDINGS:")
print("="*80)
print("""
1. BASELINE REPLICATION (Combined Data, Naive 80/20 Split + Cascading):
   - XGBoost and Random Forest use paper-aligned hyperparameters
   - 5-fold CV is applied on the training partition
   - High R² can appear due to replicate leakage under naive row-wise splitting

2. COVARIATE SHIFT (Batch 1 -> Batch 2, No TL):
   - Source-trained models tested directly on target batch
   - R² drops due to domain shift
   - Source-domain performance does not transfer perfectly to target domain

3. TRANSFER LEARNING (LightGBM Update):
   - Base model (500 trees) from Batch 1
   - Fine-tune with 20% of Batch 2 (append 100 new trees, lower lr=0.05)
   - Evaluate on held-out 80% of Batch 2
   - Demonstrates domain adaptation recovery over no-TL performance
""")
print("="*80)


SUMMARY: Paper Replication Results

Complete Results Table:
 Target                             approach        R²      RMSE       MAE
 10 1/s Baseline_XGBoost_Combined_Naive80_20  0.873784  3.391959  2.250005
  1 1/s Baseline_XGBoost_Combined_Naive80_20  0.590625 29.180476 16.430838
100 1/s Baseline_XGBoost_Combined_Naive80_20  0.875944  1.045171  0.639761
 10 1/s      Baseline_RF_Combined_Naive80_20  0.877687  3.339104  2.174583
  1 1/s      Baseline_RF_Combined_Naive80_20  0.614650 28.311260 16.143720
100 1/s      Baseline_RF_Combined_Naive80_20  0.876827  1.041442  0.611319
 10 1/s                       No_TL_Baseline  0.154989 10.553342  8.096571
  1 1/s                       No_TL_Baseline -1.197551 75.426931 56.672085
100 1/s                       No_TL_Baseline  0.540371  2.298842  1.754441
 10 1/s                   TL_LightGBM_Update  0.238449 10.355035  7.813675
  1 1/s                   TL_LightGBM_Update -1.066036 73.987556 55.104202
100 1/s                   TL_LightGBM_U

## Transfer Learning Fine-Tuning Strategy

In all transfer learning experiments, models were **first trained on Batch 1** (source domain) and **subsequently fine-tuned on Batch 2** (target domain).

### Fine-Tuning Approach

During fine-tuning, a **subset of parameters was kept fixed** to preserve previously learned representations, while only **task-specific or higher-level components were updated**. This selective parameter updating prevents catastrophic forgetting and leverages the source domain knowledge.

### Model-Specific Fine-Tuning Components

| Model | Frozen Components | Trainable Components | Learning Rate | Trees/Layers |
|-------|-------------------|----------------------|------------------|--------------|
| **LightGBM** | Original 500 trees (Batch 1) | Additional trees appended | 0.05 | +100 new trees |

### LightGBM Update Mode

LightGBM was used in **update mode**: trees learned on Batch 1 were retained and additional trees were appended using Batch 2 data. This effectively treats the original ensemble as a fixed base learner, where:
- **Base model (frozen)**: 500 trees trained on Batch 1
- **Fine-tuning addition**: 100 new trees appended using Batch 2 with lower learning rate (0.05 vs 0.10)
- **Preservation mechanism**: Original trees remain unchanged; only new trees adapt to target domain